# Load Packages

In [1]:
from glob import glob
from pathlib import Path
import warnings
import numpy as np
from pandas import read_parquet, DataFrame, concat
from scipy.stats import norm
from sklearn.metrics import mean_squared_error, r2_score

import func_gev as gev
import func_preparation as dbf
import func_plotting as dbplt
import func_utils as ut

warnings.filterwarnings("ignore", category=FutureWarning)

# Settings

In [2]:
path_input = '../input/Annual_max_DCPP_20260112/'
path_results = '../output/gev_analysis/2026-05-20/'

# Import RawData

In [3]:
ls_sim = [l for l in glob(path_input + '/*.nc')]
ls_sim

['../input/Annual_max_DCPP_20260112/Annual_max_MIROC6.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_MPI-ESM1-2-HR.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_HadGEM3-GC31-MM.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_MRI-ESM2-0.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_BCC-CSM2-MR.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_CMCC-CM2-SR5.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_CanESM5.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_NorCPM1.nc']

In [4]:
dbf.import_data_from_file(ls_sim[0])[1]

<xarray.Dataset> Size: 360MB
Dimensions:      (sample: 680, member: 2, bc: 3, sites: 11022)
Coordinates:
  * sample       (sample) int64 5kB 0 1 2 3 4 5 6 ... 674 675 676 677 678 679
  * member       (member) int64 16B 1 2
  * bc           (bc) int64 24B 1 2 3
  * sites        (sites) int64 88kB 0 1 2 3 4 ... 11017 11018 11019 11020 11021
    sim_year     (sample) float64 5kB ...
    lead         (sample) float64 5kB ...
    lon          (sites) float64 88kB ...
    lat          (sites) float64 88kB ...
    bc_flag      (sites) uint8 11kB ...
    model_valid  (sites) bool 11kB ...
Data variables:
    annualMax    (sample, member, bc, sites) float64 360MB ...
Attributes:
    description:  Modelled storm-surge annual maxima organized by simulated y...
    model:        MIROC6

# Import Results

In [4]:
raw_data = read_parquet(path_results + 'data.parquet')

In [5]:
[
    results_annual_stat_all, results_nonstat_all, location_geo_info, location_point_info
    ] = ut.import_info_for_regression(path_results)

loading ../output/gev_analysis/2026-05-20//stationary_per_year.pkl
loading ../output/gev_analysis/2026-05-20//nonstationary.pkl
loading ../output/gev_analysis/2026-05-20//LatLon.pkl
loading ../output/gev_analysis/2026-05-20//location_info.pkl


In [6]:
results_nonstat_all[0].keys()

dict_keys(['params_hat', 'n_obs', 'years_mean', 'years_std', 'Hessian eigenvalues', 'cov', 'params_std', 'return_period'])

In [7]:
results_stat_all = ut.import_pickle_data(path_results, 'stationary.pkl')

loading ../output/gev_analysis/2026-05-20//stationary.pkl


In [8]:
model_comparison_all = ut.import_pickle_data(path_results, 'model_comparison.pkl')

loading ../output/gev_analysis/2026-05-20//model_comparison.pkl


In [9]:
return_levels = read_parquet(path_results + 'return_levels.parquet')

# Model Comparison · stationary vs non-stationary preference

In [11]:
model_comparison_pvalue = [comparison['LRT']['p_value'] for comparison in model_comparison_all.values()]
df_stat_model_pvalue = DataFrame(model_comparison_pvalue)
df_stat_model_pvalue[df_stat_model_pvalue[0] < 0.05].count()

0    5201
dtype: int64

In [12]:
model_comparison_interpret = [comparison['LRT']['interpretation'] for comparison in model_comparison_all.values()]

df_stat_model_comp = DataFrame(model_comparison_interpret).value_counts()
df_stat_model_comp

0                                                                    
→ non-stationary model is significantly better \n→ μ(t) trend matters    5201
→ adding a trend doesn’t improve the fit                                 4388
Name: count, dtype: int64

In [13]:
print(
    'Non-stationary GEV was preferred at '
    f'{df_stat_model_comp.filter(like='non-stationary', axis=0).values[0] / len(model_comparison_interpret) * 100:.2f}% ' 
    'of sites'
    ) 

Non-stationary GEV was preferred at 54.24% of sites


In [14]:
for siteID, nonstat in results_nonstat_all.items():
    if nonstat['cov'] is None:
        print('covariance is None for {}')

df_nonstat_cov = [nonstat['cov'] for nonstat in results_nonstat_all.values()]

In [15]:
results_stat_all[0].get('cov')

array([[2.79605196e-05, 5.82939085e-07, 4.18532114e-07],
       [5.82939085e-07, 1.41999897e-07, 1.86618778e-08],
       [4.18532114e-07, 1.86618778e-08, 6.55128874e-08]])

In [16]:
ls_nocov_stat = []
for siteID, stat in results_stat_all.items():
    if stat.get('cov') is None:
        ls_nocov_stat.append(siteID)

#df_stat_cov = [stat['cov'] for stat in results_stat_all.values()]
len(ls_nocov_stat)

61

# Location Trend Overview

μ0: meters (m)
σ: meters (m)
μ1: meters per year (m/year)


In [10]:
import numpy as np
import pandas as pd

FACTOR_M_TO_MM = 1000

def summarize(x):
    x = pd.Series(x).replace([np.inf, -np.inf], np.nan).dropna()
    return pd.Series({
        "median": x.median(),
        "mean": x.mean(),
        "min": x.min(),
        "max": x.max(),
        "std": x.std(ddof=1),
        "p05": x.quantile(0.05),
        "p95": x.quantile(0.95),
    })

def make_parameter_summary(
    raw_data,
    results_annual_stat_all,
    results_nonstat_all,
    results_stat_all,
    model_comparison_all=None,
    return_levels=None,
):
    rows = []

    # -----------------------------
    # 1. Raw data
    # annualMax / storm_surge is in m
    if "storm_surge" in raw_data.columns:
        raw_col = "storm_surge"
    elif "annualMax" in raw_data.columns:
        raw_col = "annualMax"
    else:
        raw_col = None

    if raw_col is not None:
        s = summarize(raw_data[raw_col])
        rows.append(("Raw data", f"{raw_col}, m", s))

    # -----------------------------
    # 2. Stationary pooled GEV
    # expected params:
    # location [m], scale [m], shape [-]
    stat_records = []

    for loc_id, res in results_stat_all.items():
        if res is None:
            continue

        stat = res.get("stationary", res)

        if stat is None:
            continue

        stat_records.append({
            "location_m": stat.get("location", np.nan),
            "location_mm": stat.get("location", np.nan) * FACTOR_M_TO_MM,
            "scale_m": stat.get("scale", np.nan),
            "scale_mm": stat.get("scale", np.nan) * FACTOR_M_TO_MM,
            "shape": stat.get("shape", np.nan),
        })
    df_stat_ = pd.DataFrame(stat_records)
    df_stat = ut.mark_outliers_zmethod(df_stat_, label_col='location_m', threshold=3)

    if not df_stat.empty:
        rows.append(("Stationary GEV", "Location μ, m", summarize(df_stat["location_m"])))
        rows.append(("Stationary GEV", "Location μ, mm", summarize(df_stat["location_mm"])))
        rows.append(("Stationary GEV", "Scale σ, m", summarize(df_stat["scale_m"])))
        rows.append(("Stationary GEV", "Scale σ, mm", summarize(df_stat["scale_mm"])))
        rows.append(("Stationary GEV", "Shape ξ, dimensionless", summarize(df_stat["shape"])))

    # -----------------------------
    # 3. Non-stationary GEV
    # params_hat = [mu0, mu1, sigma, xi]
    # mu0 [m], mu1 [m/year], sigma [m], xi [-]
    ns_records = []

    for loc_id, res in results_nonstat_all.items():
        if res is None:
            continue

        ns = res.get("nonstationary", res)

        if ns is None:
            continue

        params = ns.get("params_hat", None)

        if params is None or len(params) < 4:
            continue
        
        mu0, mu1, sigma, xi = params[:4]

        ns_records.append({
            "mu0_m": mu0,
            "mu0_mm": mu0 * FACTOR_M_TO_MM,
            "mu1_m_per_yr": mu1,
            "mu1_mm_per_yr": mu1 * FACTOR_M_TO_MM,
            "sigma_m": sigma,
            "sigma_mm": sigma * FACTOR_M_TO_MM,
            "xi": xi,
        })
    df_ns_ = pd.DataFrame(ns_records)
    df_ns = ut.mark_outliers_zmethod(df_ns_, label_col='mu1_m_per_yr', threshold=3)

    if not df_ns.empty:
        rows.append(("Non-stationary GEV", "Location μ₀, m", summarize(df_ns["mu0_m"])))
        rows.append(("Non-stationary GEV", "Location μ₀, mm", summarize(df_ns["mu0_mm"])))
        rows.append(("Non-stationary GEV", "Location trend μ₁, m yr⁻¹", summarize(df_ns["mu1_m_per_yr"])))
        rows.append(("Non-stationary GEV", "Location trend μ₁, mm yr⁻¹", summarize(df_ns["mu1_mm_per_yr"])))
        rows.append(("Non-stationary GEV", "Scale σ, m", summarize(df_ns["sigma_m"])))
        rows.append(("Non-stationary GEV", "Scale σ, mm", summarize(df_ns["sigma_mm"])))
        rows.append(("Non-stationary GEV", "Shape ξ, dimensionless", summarize(df_ns["xi"])))

    # -----------------------------
    # 4. Annual stationary + regression trend
    # mu_trend has mu0 [m], mu1 [m/year]
    annual_records = []

    for loc_id, res in results_annual_stat_all.items():
        if res is None:
            continue

        trend = res.get("mu_trend", None)

        if trend is None:
            continue

        annual_records.append({
            "mu0_m": trend.get("mu0", np.nan),
            "mu0_mm": trend.get("mu0", np.nan) * FACTOR_M_TO_MM,
            "mu1_m_per_yr": trend.get("mu1", np.nan),
            "mu1_mm_per_yr": trend.get("mu1", np.nan) * FACTOR_M_TO_MM,
        })
    df_annual_ = pd.DataFrame(annual_records)
    df_annual = ut.mark_outliers_zmethod(df_annual_, label_col='mu1_m_per_yr', threshold=3)

    if not df_annual.empty:
        rows.append(("Annual-stationary regression", "Location intercept μ₀, m", summarize(df_annual["mu0_m"])))
        rows.append(("Annual-stationary regression", "Location intercept μ₀, mm", summarize(df_annual["mu0_mm"])))
        rows.append(("Annual-stationary regression", "Location trend μ₁, m yr⁻¹", summarize(df_annual["mu1_m_per_yr"])))
        rows.append(("Annual-stationary regression", "Location trend μ₁, mm yr⁻¹", summarize(df_annual["mu1_mm_per_yr"])))
        
        rows.append(("Annual-stationary regression", "Scale σ, m", summarize(df_annual["mu0_m"])))
        rows.append(("Annual-stationary regression", "Scale σ, mm", summarize(df_annual["mu0_mm"])))
        rows.append(("Annual-stationary regression", "Shape ξ, dimensionless", summarize(df_annual["mu1_m_per_yr"])))

    # Final table
    summary = pd.DataFrame([
        {
            "approach": approach,
            "metric": metric,
            "median": stats["median"],
            "mean": stats["mean"],
            "min": stats["min"],
            "max": stats["max"],
            "std": stats["std"],
            "p05": stats["p05"],
            "p95": stats["p95"],
        }
        for approach, metric, stats in rows
    ])

    return summary, df_stat, df_ns, df_annual


summary_table, df_stat_params, df_nonstat_params, df_annual_trend_params = make_parameter_summary(
    raw_data=raw_data,
    results_annual_stat_all=results_annual_stat_all,
    results_nonstat_all=results_nonstat_all,
    results_stat_all=results_stat_all,
    model_comparison_all=model_comparison_all,
    return_levels=return_levels,
)

summary_table

Marked 126 outliers using modified Z-score method
Marked 89 outliers using modified Z-score method
Marked 172 outliers using modified Z-score method


,approach,metric,median,mean,min,max,std,p05,p95
0,Raw data,"storm_surge, m",0.388854,0.566502,-0.248328,4.218655,0.416490,0.168238,1.363128
1,Stationary GEV,"Location μ, m",0.422817,0.555714,0.117840,2.160527,0.383311,0.183431,1.258435
2,Stationary GEV,"Location μ, mm",422.816553,555.713573,117.840107,2160.526918,383.311234,183.431277,1258.434552
3,Stationary GEV,"Scale σ, m",0.081758,0.109109,0.025352,0.494889,0.074383,0.038518,0.241280
4,Stationary GEV,"Scale σ, mm",81.758192,109.109311,25.352065,494.889207,74.382786,38.518007,241.280433
5,Stationary GEV,"Shape ξ, dimensionless",-0.138860,-0.129140,-0.538234,0.148790,0.068003,-0.221736,-0.004917
6,Non-stationary GEV,"Location μ₀, m",0.423413,0.557729,0.117837,2.176620,0.382563,0.187149,1.259864
7,Non-stationary GEV,"Location μ₀, mm",423.412632,557.729206,117.837234,2176.619989,382.562931,187.149497,1259.863872
8,Non-stationary GEV,"Location trend μ₁, m yr⁻¹",-0.000050,-0.000015,-0.001955,0.001544,0.000203,-0.000267,0.000312
9,Non-stationary GEV,"Location trend μ₁, mm yr⁻¹",-0.049640,-0.015020,-1.954687,1.543792,0.202647,-0.267094,0.312044


In [11]:
stat_gev = summary_table[summary_table['approach'] == 'Annual-stationary regression']

ls_metric = ['Location intercept μ₀, m', 'Location trend μ₁, mm yr⁻¹', 'Scale σ, mm', 'Shape ξ, dimensionless']
ls = []
for i in stat_gev.index:
    if stat_gev.loc[i, 'metric'] in ls_metric:
        ls.append(pd.DataFrame(stat_gev.loc[i]))

concat(ls, axis=1).T

,approach,metric,median,mean,min,max,std,p05,p95
13,Annual-stationary regression,"Location intercept μ₀, m",0.4243,0.54707,0.118009,2.233776,0.373472,0.180578,1.247413
16,Annual-stationary regression,"Location trend μ₁, mm yr⁻¹",-0.050282,0.046263,-4.877693,4.914024,0.527688,-0.612028,0.986064
18,Annual-stationary regression,"Scale σ, mm",424.299787,547.069698,118.008855,2233.775605,373.471783,180.577689,1247.412763
19,Annual-stationary regression,"Shape ξ, dimensionless",-0.00005,0.000046,-0.004878,0.004914,0.000528,-0.000612,0.000986


## Stationary

In [24]:
df_stat = pd.DataFrame(
    [(stat['location'], stat['scale'], stat['shape']) for stat in results_stat_all.values()], 
    columns=['location', 'scale', 'shape']
    )

label_para = 'location'
para_stat_outlier = ut.mark_outliers_zmethod(df_stat, label_col=label_para, threshold=3)


Marked 126 outliers using modified Z-score method


In [25]:
para_stat_outlier.describe().T

,count,mean,std,min,25%,50%,75%,max
location,9589.0,0.555714,0.383311,0.117840,0.235264,0.422817,0.812151,2.160527
scale,9589.0,0.109109,0.074383,0.025352,0.048649,0.081758,0.156626,0.494889
shape,9589.0,-0.129140,0.068003,-0.538234,-0.174587,-0.138860,-0.093972,0.148790


In [30]:
100 * (para_stat_outlier["shape"] < 0).sum() / len(para_stat_outlier)

np.float64(95.99541140890604)

## Non-stationary

In [31]:
df_results_nonstat = DataFrame([results_nonstat['params_hat'] for results_nonstat in results_nonstat_all.values()])
df_results_nonstat.columns = ['loc', 'loc_trend','scale', 'shape']

df_results_nonstat['loc_trend_mm/yr'] = df_results_nonstat['loc_trend']*1000

##### outlier removal

In [32]:
label_para = 'loc_trend_mm/yr'

para_nonstat_outlier = ut.mark_outliers_zmethod(df_results_nonstat, label_col=label_para, threshold=3)
para_nonstat_outlier = para_nonstat_outlier[para_nonstat_outlier.outliers == False]

if label_para == 'loc_trend' or label_para == 'scale':
    para_nonstat_outlier = para_nonstat_outlier*1000
    
print(f'median: {para_nonstat_outlier[label_para].median():.3f}')
print(f"min:\t{para_nonstat_outlier[label_para].describe()['min']:.3f}")
print(f"max:\t{para_nonstat_outlier[label_para].describe()['max']:.3f}")
print(f"STD:\t{para_nonstat_outlier[label_para].describe()['std']:.3f}")

para_nonstat_outlier.describe().T

Marked 89 outliers using modified Z-score method
median: -0.049
min:	-0.622
max:	0.584
STD:	0.171


,count,mean,std,min,25%,50%,75%,max
loc,9500.0,0.549932,0.374646,0.117837,0.238779,0.418789,0.810060,2.152442
loc_trend,9500.0,-0.000014,0.000171,-0.000622,-0.000106,-0.000049,0.000060,0.000584
scale,9500.0,0.104518,0.067913,0.025183,0.047247,0.079597,0.152161,0.416195
shape,9500.0,-0.154218,0.071881,-0.563239,-0.200606,-0.164699,-0.118420,0.150610
loc_trend_mm/yr,9500.0,-0.014430,0.170745,-0.621920,-0.105663,-0.049454,0.059747,0.584140


In [33]:
df_results_nonstat.describe().T

,count,mean,std,min,25%,50%,75%,max
loc,9589.0,0.557729,0.382563,0.117837,0.239021,0.423413,0.821244,2.176620
loc_trend,9589.0,-0.000015,0.000203,-0.001955,-0.000106,-0.000050,0.000061,0.001544
scale,9589.0,0.105931,0.069409,0.025183,0.047330,0.081101,0.153755,0.416195
shape,9589.0,-0.154046,0.072003,-0.563239,-0.200626,-0.164538,-0.117751,0.150610
loc_trend_mm/yr,9589.0,-0.015020,0.202647,-1.954687,-0.106485,-0.049640,0.061082,1.543792


In [34]:
100 * (df_results_nonstat["shape"] < 0).sum() / len(df_results_nonstat)

np.float64(95.99541140890604)

### statistics

In [199]:
df_mu1_pos = df_results_nonstat[df_results_nonstat.loc_trend >0].loc_trend.count() / df_results_nonstat.shape[0]* 100
df_mu1_pos_outliers = para_nonstat_outlier[para_nonstat_outlier.loc_trend >0].loc_trend.count() / para_nonstat_outlier.shape[0]* 100

df_mu1_pos, df_mu1_pos_outliers

(np.float64(35.29043695901554), np.float64(35.26315789473684))

In [237]:
site_loc_trend_max = df_results_nonstat['loc_trend_mm/yr'].idxmin()
(
    df_results_nonstat.loc[site_loc_trend_max, 'loc_trend_mm/yr'], 
    location_point_info[site_loc_trend_max], 
    location_geo_info[site_loc_trend_max]
)

(np.float64(-1.9546872858242077),
 'Busum Schleswig-Holstein DE',
 (np.float64(54.11875930770386), np.float64(8.783789375795145)))

In [236]:
site_loc_trend_max_outlier = para_nonstat_outlier['loc_trend_mm/yr'].idxmin()
print(site_loc_trend_max_outlier)

(
    para_nonstat_outlier.loc[site_loc_trend_max_outlier, 'loc_trend_mm/yr'], 
    location_point_info[site_loc_trend_max_outlier],
    location_geo_info[site_loc_trend_max_outlier]
)

4271


(np.float64(-0.6219200095859199),
 'Embleton England GB',
 (np.float64(55.46231382844708), np.float64(-1.5884571683002715)))

In [202]:
print(
    f'Positive μ₁ trends were identified at {df_mu1_pos:.2f}% of sites ({df_mu1_pos_outliers:.2f}% considering outliers),'
    f'\nwith the strongest trends concentrated along {location_point_info[site_loc_trend_max]}'
    )

Positive μ₁ trends were identified at 35.29% of sites (35.26% considering outliers),
with the strongest trends concentrated along Arnside England GB


##  Annual-stationary

In [35]:
df_mu1_astat_all = DataFrame(
    [astat['mu_trend']['mu1'] for  astat in results_annual_stat_all.values()], 
    index=results_annual_stat_all.keys(), columns=['loc_trend']
    )
df_mu1_astat_all['loc_trend_mm/yr'] = df_mu1_astat_all['loc_trend']*1000


df_mu1_astat_all = pd.concat([
    df_mu1_astat_all, 
    pd.DataFrame([
        annual_stat['annual_mle'][['location', 'scale', 'shape']].mean() 
        for annual_stat in results_annual_stat_all.values()
        ])
    ], axis=1)
df_mu1_astat_all

,loc_trend,loc_trend_mm/yr,location,scale,shape
0,-0.000077,-0.076743,0.125993,0.031448,-0.075392
1,-0.000058,-0.057600,0.126412,0.030983,-0.116212
2,-0.000056,-0.056471,0.126116,0.030841,-0.115840
3,-0.000066,-0.066158,0.126220,0.031078,-0.101798
4,0.000109,0.109255,0.128779,0.043621,-0.168116
...,...,...,...,...,...
9584,-0.000443,-0.442955,0.227887,0.102146,-0.611190
9585,-0.000212,-0.212398,0.297875,0.051158,-0.349188
9586,-0.000094,-0.094383,0.301296,0.064511,-0.358677
9587,-0.000224,-0.223714,0.302621,0.067770,-0.379535


#### outlier removal

In [36]:
label_para = 'loc_trend_mm/yr'

para_astat_outlier = ut.mark_outliers_zmethod(df_mu1_astat_all, label_col=label_para, threshold=3)
para_astat_outlier = para_astat_outlier[para_astat_outlier.outliers == False]

if label_para == 'loc_trend' or label_para == 'scale':
    para_astat_outlier = para_astat_outlier*1000
    
print(f'median: {para_astat_outlier[label_para].median():.3f}')
print(f"min:\t{para_astat_outlier[label_para].describe()['min']:.3f}")
print(f"max:\t{para_astat_outlier[label_para].describe()['max']:.3f}")
print(f"STD:\t{para_astat_outlier[label_para].describe()['std']:.3f}")

para_astat_outlier.describe().T

Marked 172 outliers using modified Z-score method
median: -0.050
min:	-1.528
max:	1.626
STD:	0.431


,count,mean,std,min,25%,50%,75%,max
loc_trend,9417.0,0.000053,0.000431,-0.001528,-0.000159,-0.000050,0.000235,0.001626
loc_trend_mm/yr,9417.0,0.053023,0.430832,-1.528384,-0.159371,-0.049927,0.235303,1.625559
location,9417.0,0.536881,0.367072,0.118009,0.237569,0.414275,0.775254,2.190255
scale,9417.0,0.126356,0.082341,0.026005,0.064337,0.094288,0.183881,0.830480
shape,9417.0,-0.222619,0.189846,-0.903992,-0.331597,-0.199233,-0.092135,0.434608


In [37]:
100 * (para_astat_outlier["shape"] < 0).sum() / len(para_astat_outlier)

np.float64(89.85876606137836)

### statistics

In [21]:
df_mu1_astat_pos = para_astat_outlier[para_astat_outlier.loc_trend >0].loc_trend.count() / para_astat_outlier.shape[0]* 100
df_mu1_astat_pos_outliers = para_astat_outlier[para_astat_outlier.loc_trend >0].loc_trend.count() / para_astat_outlier.shape[0]* 100

df_mu1_astat_pos, df_mu1_astat_pos_outliers

(np.float64(43.060422639906555), np.float64(43.060422639906555))

In [22]:
site_loc_trend_max = df_mu1_astat_all['loc_trend_mm/yr'].idxmax()
print('the most positive trend in the annual-stationary regression is at site', site_loc_trend_max)
(
    df_mu1_astat_all.loc[site_loc_trend_max, 'loc_trend_mm/yr'], 
    location_point_info[site_loc_trend_max], 
    location_geo_info[site_loc_trend_max]
)

the most positive trend in the annual-stationary regression is at site 5530


(np.float64(4.914024081096653),
 'West-Terschelling Friesland NL',
 (np.float64(53.21822350627164), np.float64(5.253820469013097)))

In [23]:
site_loc_trend_max = para_astat_outlier['loc_trend_mm/yr'].idxmax()
print('the most positive trend in the annual-stationary regression is at site (with outlier removal)', site_loc_trend_max)
(
    para_astat_outlier.loc[site_loc_trend_max, 'loc_trend_mm/yr'], 
    location_point_info[site_loc_trend_max], 
    location_geo_info[site_loc_trend_max]
)

the most positive trend in the annual-stationary regression is at site (with outlier removal) 5957


(np.float64(1.6255592390601228),
 'Harboore Central Jutland DK',
 (np.float64(56.46405982078649), np.float64(8.124438719530673)))

In [24]:
site_loc_trend_max = df_mu1_astat_all['loc_trend_mm/yr'].idxmin()
print('the most negative trend in the annual-stationary regression is at site', site_loc_trend_max)

(
    df_mu1_astat_all.loc[site_loc_trend_max, 'loc_trend_mm/yr'], 
    location_point_info[site_loc_trend_max], 
    location_geo_info[site_loc_trend_max]
)

the most negative trend in the annual-stationary regression is at site 5994


(np.float64(-4.877693354206012),
 'Thyboron Central Jutland DK',
 (np.float64(56.77709359746483), np.float64(8.229638913218421)))

In [25]:
site_loc_trend_max = para_astat_outlier['loc_trend_mm/yr'].idxmin()
print('the most negative trend in the annual-stationary regression is at site (with outlier removal)', site_loc_trend_max)
(
    para_astat_outlier.loc[site_loc_trend_max, 'loc_trend_mm/yr'], 
    location_point_info[site_loc_trend_max], 
    location_geo_info[site_loc_trend_max]
)

the most negative trend in the annual-stationary regression is at site (with outlier removal) 4791


(np.float64(-1.5283835664864223),
 'Walton-on-the-Naze England GB',
 (np.float64(51.869324800488485), np.float64(1.2944004765110506)))

## Comparison

In [38]:
mu1_astat_all = df_mu1_astat_all['loc_trend_mm/yr'].values
mu1_ns_all = df_results_nonstat['loc_trend_mm/yr'].values

rmse = np.sqrt(mean_squared_error(mu1_ns_all, mu1_astat_all))
r2 = r2_score(mu1_ns_all, mu1_astat_all)

print(f'RMSE: {rmse:.4f}')
print(f'R²: {r2:.4f}')
print(f'Mean Absolute Error (non-stationary): {np.mean(np.abs(mu1_ns_all)):.4f} mm/year')
print(f'Mean Absolute Error (annual-stationary): {np.mean(np.abs(mu1_astat_all)):.4f} mm/year')

RMSE: 0.4722
R²: -4.4298
Mean Absolute Error (non-stationary): 0.1379 mm/year
Mean Absolute Error (annual-stationary): 0.3370 mm/year


In [39]:
df_comparison = concat([
    para_nonstat_outlier['loc_trend_mm/yr'], para_astat_outlier['loc_trend_mm/yr']
    ], axis=1).dropna()
df_comparison.columns = ['loc_trend_nonstat', 'loc_trend_astat']

df_comparison.describe()

,loc_trend_nonstat,loc_trend_astat
count,9351.000000,9351.000000
mean,-0.013788,0.052333
std,0.169434,0.427263
min,-0.621920,-1.528384
25%,-0.105227,-0.159200
50%,-0.049638,-0.050146
75%,0.058927,0.233289
max,0.584140,1.625559


In [40]:
mu1_astat_all = df_comparison.loc_trend_astat.values
mu1_ns_all = df_comparison.loc_trend_nonstat.values

rmse = np.sqrt(mean_squared_error(mu1_ns_all, mu1_astat_all))
r2 = r2_score(mu1_ns_all, mu1_astat_all)

print(f'RMSE: {rmse:.3f}')
print(f'R²: {r2:.3f}')
print(f'Mean Absolute Error (non-stationary): {np.median(np.abs(mu1_ns_all)):.4f} mm/year')
print(f'Mean Absolute Error (annual-stationary): {np.median(np.abs(mu1_astat_all)):.4f} mm/year')

RMSE: 0.375
R²: -3.894
Mean Absolute Error (non-stationary): 0.0965 mm/year
Mean Absolute Error (annual-stationary): 0.1818 mm/year


# Return Period Overview

In [ ]:
def check_return_period_consistency(nonstat, ref_year=1961, future_year=2026, T_ref=50):
    mu0, mu1, sigma, xi = nonstat["params_hat"]
    years_mean = nonstat["years_mean"]

    mu_ref = mu0 + mu1 * (ref_year - years_mean)
    z_ref = genextreme.isf(1 / T_ref, c=-xi, loc=mu_ref, scale=sigma)

    rp_ref = 1 / genextreme.sf(z_ref, c=-xi, loc=mu_ref, scale=sigma)

    mu_future = mu0 + mu1 * (future_year - years_mean)
    rp_future = 1 / genextreme.sf(z_ref, c=-xi, loc=mu_future, scale=sigma)

    print("mu_ref:", mu_ref)
    print("mu_future:", mu_future)
    print("delta_mu mm:", (mu_future - mu_ref) * 1000)
    print("z_ref mm:", z_ref * 1000)
    print("RP at ref year:", rp_ref)
    print("RP at future year:", rp_future)

In [ ]:
return_levels_50yr = return_levels[return_levels.return_period == 50]
return_levels_50yr = return_levels_50yr[return_levels_50yr.t_eval == 1961]

# a 50year event in 1961 
return_levels_50yr

In [ ]:
return_levels_50yr_stat = return_levels_50yr[return_levels_50yr.model == 'stationary'][['z_T','lower', 'upper']].median()
return_levels_50yr_stat_m = return_levels_50yr_stat*1000 # meter

In [ ]:
return_levels_50yr_nstat = return_levels_50yr[return_levels_50yr.model == 'nonstationary'][['z_T','lower', 'upper']].median()
return_levels_50yr_nstat_m = return_levels_50yr_nstat*1000 # meter

In [ ]:
return_levels_50yr_stat_m

In [ ]:
print(
    'The median 50-year return level across all sites was\n',
    f'stationary approach: {return_levels_50yr_stat_m.z_T:.2f}m (95% CI: {return_levels_50yr_stat_m.lower:.2f}m - {return_levels_50yr_stat_m.upper:.2f}m)\n',
    f'non-stationary approach: {return_levels_50yr_nstat_m.z_T:.2f}m (95% CI: {return_levels_50yr_nstat_m.lower:.2f}m - {return_levels_50yr_nstat_m.upper:.2f}m)\n',  
)

# Return Levels

In [57]:
t_eval = 1961

for approach in ['stationary', 'nonstationary']:
    print(f'Summary {approach} approach:')
    for return_period in [10, 50, 100]:
        return_level_approach = return_levels[return_levels.model == approach]

        return_level_approach_teval = return_level_approach[return_level_approach.t_eval == t_eval]
        df = return_level_approach_teval[return_level_approach_teval['return_period'] == return_period]

        q25 = df[['z_T', 'lower', 'upper']].describe().loc['25%']
        q75 = df[['z_T', 'lower', 'upper']].describe().loc['75%']
        median_ = df[['z_T', 'lower', 'upper']].median().T.loc['z_T']

        iqr = q75 - q25
        ci_width = (df['upper'] - df['lower']).median()
            
        print(
            f"\tT={return_period}: median = {median_:.3f} m "
            f"| iqr = {iqr['z_T']:.3f} m "
            f"| median 90%CI width={ci_width:.3f} m"
            )
        

Summary stationary approach:
	T=10: median = 0.585 m | iqr = 0.794 m | median 90%CI width=0.030 m
	T=50: median = 0.689 m | iqr = 0.910 m | median 90%CI width=0.034 m
	T=100: median = 0.730 m | iqr = 0.952 m | median 90%CI width=0.036 m
Summary nonstationary approach:
	T=10: median = 0.582 m | iqr = 0.782 m | median 90%CI width=0.064 m
	T=50: median = 0.678 m | iqr = 0.881 m | median 90%CI width=0.095 m
	T=100: median = 0.715 m | iqr = 0.913 m | median 90%CI width=0.107 m


In [59]:
site_id = df.loc[df['upper'].idxmin()].loc['location_id']
location_point_info[site_id]

'Las Palmas de Gran Canaria Canary Islands ES'

# Fit Parameters

## stationary

In [28]:
para_stat = concat([DataFrame([[
    result_stat['location'], result_stat['location_std'],
    result_stat['scale'], result_stat['scale_std'],
    result_stat['shape'], result_stat['shape_std']]
], index=[siteID]) for siteID, result_stat in results_stat_all.items()])

para_stat.columns = ['loc', 'loc_std', 'scale', 'scale_std', 'shape', 'shape_std']

#### outlier removal and description

In [ ]:
label_para = 'scale'

para_stat_outlier = ut.mark_outliers_zmethod(para_stat, label_col=label_para, threshold=3)
para_stat_outlier = para_stat_outlier[para_stat_outlier.outliers == False]

if label_para == 'loc_trend' or label_para == 'scale':
    para_stat_outlier = para_stat_outlier*1000
    
print(f'median: {para_stat_outlier[label_para].median():.3f}')
print(f"min:\t{para_stat_outlier[label_para].describe()['min']:.3f}")
print(f"max:\t{para_stat_outlier[label_para].describe()['max']:.3f}")
print(f"STD:\t{para_stat_outlier[label_para].describe()['std']:.3f}")

para_stat_outlier.describe()

In [ ]:
para_stat[para_stat['shape'] < 0]['shape'].count() / para_stat.shape[0] * 100

## non-stationary

In [ ]:
para_nonstat = concat([
    DataFrame(nonstat['params_hat'], columns=[siteID]).T 
    for siteID, nonstat in results_nonstat_all.items()
    ])
para_nonstat.columns = ['loc', 'loc_trend', 'scale', 'shape']

### outlier removal and description

In [ ]:
label_para = 'loc_trend'

para_nonstat_outlier = ut.mark_outliers_zmethod(para_nonstat, label_col=label_para, threshold=3)
para_nonstat_outlier = para_nonstat_outlier[para_nonstat_outlier.outliers == False]

if label_para == 'loc_trend' or label_para == 'scale':
    para_nonstat_outlier = para_nonstat_outlier*1000
    
print(f'median: {para_nonstat_outlier[label_para].median():.3f}')
print(f"min:\t{para_nonstat_outlier[label_para].describe()['min']:.3f}")
print(f"max:\t{para_nonstat_outlier[label_para].describe()['max']:.3f}")
print(f"STD:\t{para_nonstat_outlier[label_para].describe()['std']:.3f}")

para_nonstat_outlier.describe()

### statistics

In [ ]:
n_total_sites = para_nonstat_outlier.shape[0]

In [ ]:
para_nonstat_outlier[para_nonstat_outlier['shape'] < 0]['shape'].count() / n_total_sites * 100

In [ ]:
para_nonstat_outlier[para_nonstat_outlier.loc_trend > 0].loc_trend.count() / n_total_sites * 100

In [ ]:
# Wald statistic to p-value in location trend (two-sided)
z_stat = para_nonstat_outlier.loc_trend / para_nonstat_outlier.loc_trend.std()          
pvalue = 2 * (1 - norm.cdf(abs(z_stat))) 

(pvalue < 0.05).mean() * 100

In [ ]:
location_geo_info[para_nonstat_outlier.loc_trend.idxmax()], location_point_info[para_nonstat_outlier.loc_trend.idxmax()]

In [ ]:
location_geo_info[para_nonstat_outlier.loc_trend.idxmin()], location_point_info[para_nonstat_outlier.loc_trend.idxmin()]

In [ ]:
para_nonstat_outlier.loc_trend.max(), para_nonstat_outlier.loc_trend.min()

## annual-stationary

In [ ]:
para_astat = concat([
    concat([astat['annual_mle'][['location', 'scale', 'shape']].median() 
            for astat in results_annual_stat_all.values()], axis=1).T,
    DataFrame([
        astat['mu_trend']['mu1'] 
        for astat in results_annual_stat_all.values()], columns=['mu1'])
    ], axis=1)

para_astat.columns = ['loc', 'scale', 'shape', 'loc_trend']
para_astat

### outlier removal and description

In [ ]:
label_para = 'loc_trend'

para_astat_outlier = ut.mark_outliers_zmethod(para_astat, label_col=label_para, threshold=3)
para_astat_outlier = para_astat_outlier[para_astat_outlier.outliers == False]

if label_para == 'loc_trend' or label_para == 'scale':
    para_astat_outlier = para_astat_outlier*1000
    
print(f'median: {para_astat_outlier[label_para].median():.3f}')
print(f"min:\t{para_astat_outlier[label_para].describe()['min']:.3f}")
print(f"max:\t{para_astat_outlier[label_para].describe()['max']:.3f}")
print(f"STD:\t{para_astat_outlier[label_para].describe()['std']:.3f}")

para_astat_outlier.describe()

### statistics

In [40]:
n_total_sites_astat = para_astat_outlier.shape[0]

NameError: name 'para_astat_outlier' is not defined

In [ ]:
para_astat_outlier[para_astat_outlier['shape'] < 0]['shape'].count() / n_total_sites_astat * 100

In [ ]:
para_astat_outlier[para_astat_outlier.loc_trend > 0].loc_trend.count() / n_total_sites_astat * 100

In [ ]:
# Wald statistic to p-value in location trend (two-sided)
z_stat = para_astat_outlier.loc_trend / para_astat_outlier.loc_trend.std()          
pvalue = 2 * (1 - norm.cdf(abs(z_stat))) 

(pvalue < 0.05).mean() * 100

In [ ]:
print('maximum location trend:',
    location_geo_info[para_astat_outlier.loc_trend.idxmax()], 
    location_point_info[para_astat_outlier.loc_trend.idxmax()], 
    para_astat_outlier.loc_trend.max()
)

In [ ]:
print('minimum location trend:',
    location_geo_info[para_astat_outlier.loc_trend.idxmin()], 
    location_point_info[para_astat_outlier.loc_trend.idxmin()], 
    para_astat_outlier.loc_trend.min()
)

# Goodness-of-Fit

In [ ]:
import matplotlib.pyplot as plt

def plot_gev_qq(data, mu, sigma, xi, ax=None):
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import genextreme

    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))

    x = np.sort(np.asarray(data))
    n = len(x)
    p = (np.arange(1, n + 1) - 0.5) / n

    q = genextreme.ppf(p, c=-xi, loc=mu, scale=sigma)

    ax.scatter(q, x, s=12, alpha=0.6)
    lims = [min(q.min(), x.min()), max(q.max(), x.max())]
    ax.plot(lims, lims, "k--", lw=1)

    ax.set_xlabel("Theoretical GEV quantiles")
    ax.set_ylabel("Observed annual maxima")
    ax.set_title("GEV QQ plot")
    return ax


def plot_gev_pp(data, mu, sigma, xi, ax=None):
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import genextreme

    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))

    x = np.sort(np.asarray(data))
    n = len(x)

    empirical_p = (np.arange(1, n + 1) - 0.5) / n
    fitted_p = genextreme.cdf(x, c=-xi, loc=mu, scale=sigma)

    ax.scatter(fitted_p, empirical_p, s=12, alpha=0.6)
    ax.plot([0, 1], [0, 1], "k--", lw=1)

    ax.set_xlabel("Fitted GEV probability")
    ax.set_ylabel("Empirical probability")
    ax.set_title("GEV PP plot")
    return ax


def plot_return_level_diagnostic(data, mu, sigma, xi, ax=None):
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import genextreme

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4))

    x = np.sort(np.asarray(data))
    n = len(x)

    p_emp = (np.arange(1, n + 1) - 0.5) / n
    T_emp = 1 / (1 - p_emp)

    T_grid = np.array([2, 5, 10, 25, 50, 100, 200])
    z_grid = genextreme.isf(1 / T_grid, c=-xi, loc=mu, scale=sigma)

    ax.scatter(T_emp, x, s=12, alpha=0.5, label="Empirical")
    ax.plot(T_grid, z_grid, "k-", label="Fitted GEV")

    ax.set_xscale("log")
    ax.set_xlabel("Return period, years")
    ax.set_ylabel("Return level")
    ax.set_title("Return-level diagnostic")
    ax.legend()
    return ax

In [65]:
loc_id = 0

res = results_stat_all[loc_id]

mu = res["location"]
sigma = res["scale"]
xi = res["shape"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_gev_qq(raw_data, mu, sigma, xi, ax=axes[0])
plot_gev_pp(raw_data, mu, sigma, xi, ax=axes[1])
plot_return_level_diagnostic(raw_data, mu, sigma, xi, ax=axes[2])

plt.tight_layout()
plt.show()

TypeError: '<' not supported between instances of 'int' and 'str'